# Day 4: Tools — The Day My Chatbot Became Useful

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Without tools, an agent can only talk. With tools, it can act.

An LLM without tools is like a consultant who can only give advice — useful, but
limited. Today we give our agents **tools** (calculator, search) and watch them
**decide when to use them**.

This is the most important concept in agent development. Tools are what separate a
chatbot from an agent.

## What You'll Build
- An agent with calculator and search tools
- The same agent in all 3 frameworks
- Watch how each framework handles tool calling differently

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 4 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 4 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Questions

1. **"What is 15% of 2340?"** — requires the calculator tool
2. **"What is the capital of Australia?"** — requires the search tool
3. **"Calculate 25% of 4800 and tell me the capital of Brazil."** — requires BOTH tools

The agent must **decide** which tool to use. This is what makes it an agent, not a chatbot.

---
## Framework 1: OpenAI Agents SDK

Tools use the `@function_tool` decorator. The docstring becomes the tool description
that the agent reads to decide when to use it.

In [3]:
from agents import Agent, Runner, function_tool

model = get_openai_agents_model()

# ── Define tools ───────────────────────────────────────
@function_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Handles percentages like '15% of 2340'."""
    import re
    pct_match = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct_match:
        return str(float(pct_match.group(2)) * float(pct_match.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

@function_tool
def search(query: str) -> str:
    """Search a knowledge base for geography facts, AI concepts, and tech topics."""
    knowledge = {
        "capital of australia": "Canberra",
        "capital of brazil": "Brasilia",
        "capital of france": "Paris",
        "capital of japan": "Tokyo",
        "capital of india": "New Delhi",
        "capital of germany": "Berlin",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"No results for: {query}"

# ── Create agent with tools ─────────────────────────────
agent = Agent(
    name="Tool Assistant",
    instructions="You have a calculator and a search tool. Use them when needed. Always show your work.",
    tools=[calculate, search],
    model=model,
)

# ── Test questions ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Australia?",
    "Calculate 25% of 4800 and tell me the capital of Brazil.",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"Q: {q}")
    print(f"{'=' * 60}")
    result = result = await Runner.run(agent, q)
    print(result.final_output)


Q: What is 15% of 2340?
15% of 2340 is 351.0.

Q: What is the capital of Australia?
The capital of Australia is Canberra.

Q: Calculate 25% of 4800 and tell me the capital of Brazil.
25% of 4800 is 1200. The capital of Brazil is Brasília.


---
## Framework 2: LangGraph

LangGraph tools use `@tool` from `langchain_core`. The `create_react_agent` prebuilt
automatically handles the tool-calling loop: agent reasons → calls tool → agent reasons
with tool result → final answer.

In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

model = get_langgraph_model()

# ── Define tools ───────────────────────────────────────
@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Handles percentages like '15% of 2340'."""
    import re
    pct_match = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct_match:
        return str(float(pct_match.group(2)) * float(pct_match.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

@tool
def search(query: str) -> str:
    """Search a knowledge base for geography facts, AI concepts, and tech topics."""
    knowledge = {
        "capital of australia": "Canberra",
        "capital of brazil": "Brasilia",
        "capital of france": "Paris",
        "capital of japan": "Tokyo",
        "capital of india": "New Delhi",
        "capital of germany": "Berlin",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"No results for: {query}"

# ── Create agent with tools ─────────────────────────────
agent = create_react_agent(
    model,
    tools=[calculate, search],
    prompt="You have a calculator and a search tool. Use them when needed. Always show your work.",
)

# ── Test questions ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Australia?",
    "Calculate 25% of 4800 and tell me the capital of Brazil.",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"Q: {q}")
    print(f"{'=' * 60}")
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    # Show the full message trace
    for msg in result["messages"]:
        role = getattr(msg, 'type', 'unknown')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            print(f"  [TOOL CALL] {msg.tool_calls}")
        elif role == 'tool':
            print(f"  [TOOL RESULT] {msg.content[:100]}")
        elif role != 'system':
            content = msg.content if hasattr(msg, 'content') else str(msg)
            if content:
                print(f"  [{role.upper()}] {content[:200]}")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_23914/1770402896.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



Q: What is 15% of 2340?
  [HUMAN] What is 15% of 2340?
  [TOOL CALL] [{'name': 'calculate', 'args': {'expression': '15% of 2340'}, 'id': '22b6b093-f82c-4ec3-9365-1d1983ff7d56', 'type': 'tool_call'}]
  [TOOL RESULT] 351.0
  [AI] 15% of 2340 is 351.0.

Q: What is the capital of Australia?
  [HUMAN] What is the capital of Australia?
  [TOOL CALL] [{'name': 'search', 'args': {'query': 'capital of Australia'}, 'id': '6301ef45-b666-4516-9dd0-69621da60b17', 'type': 'tool_call'}]
  [TOOL RESULT] Canberra
  [AI] The capital of Australia is Canberra.

Q: Calculate 25% of 4800 and tell me the capital of Brazil.
  [HUMAN] Calculate 25% of 4800 and tell me the capital of Brazil.
  [TOOL CALL] [{'name': 'calculate', 'args': {'expression': '25% of 4800'}, 'id': '98415769-24ae-4e63-b949-435805878941', 'type': 'tool_call'}, {'name': 'search', 'args': {'query': 'capital of Brazil'}, 'id': '14374c46-fca6-42e6-996d-91e4f8c826e7', 'type': 'tool_call'}]
  [TOOL RESULT] 1200.0
  [TOOL RESULT] Brasilia
  [AI

---
## Framework 3: CrewAI

CrewAI handles tools differently — tools can be given to the Agent or the Crew.
The agent decides when to use them based on the task description.

In [8]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

llm = get_crewai_llm()

# ── Define tools (CrewAI uses its own @tool decorator) ────
@crewai_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Handles percentages like '15% of 2340'."""
    import re
    pct_match = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct_match:
        return str(float(pct_match.group(2)) * float(pct_match.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

@crewai_tool
def search(query: str) -> str:
    """Search a knowledge base for geography facts, AI concepts, and tech topics."""
    knowledge = {
        "capital of australia": "Canberra",
        "capital of brazil": "Brasilia",
        "capital of france": "Paris",
        "capital of japan": "Tokyo",
        "capital of india": "New Delhi",
        "capital of germany": "Berlin",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"No results for: {query}"

# ── Create agent with tools ─────────────────────────────
assistant = Agent(
    role="Tool Assistant",
    goal="Answer questions using calculator and search tools",
    backstory="You are a precise assistant who always uses the right tool for the job.",
    tools=[calculate, search],
    llm=llm, verbose=True,
)

# ── Test questions ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Australia?",
    "Calculate 25% of 4800 and tell me the capital of Brazil.",
]

for q in questions:
    task = Task(
        description=q,
        expected_output="The answer using tools.",
        agent=assistant,
    )
    crew = Crew(agents=[assistant], tasks=[task], process=Process.sequential, verbose=True)
    result = await crew.kickoff_async()
    print(f"\n{'=' * 60}")
    print(f"Q: {q}")
    print(f"{'=' * 60}")
    print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5dfed2ea-234a-4da0-b0e0-d07079732169                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is 15% of 2340?                                                                                     │
│  ID: c02b35c0-827e-40d7-9a77-702841ef60bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tool Assistant                                                                                          │
│                                                                                                                 │
│  Task: What is 15% of 2340?                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculate executed with result: 351.0...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tool Assistant                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The answer to the task is that 25% of 4800 is 1200 and the capital of Brazil is Brasília.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Calculate 25% of 4800 and tell me the capital of Brazil.                                                 │
│  Agent: Tool Assistant                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 94cf8cae-a6bf-425f-b091-969b69558044                                                                       │
│  Final Output: The answer to the task is that 25% of 4800 is 1200 and the capital of Brazil is Brasília.        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: Calculate 25% of 4800 and tell me the capital of Brazil.
The answer to the task is that 25% of 4800 is 1200 and the capital of Brazil is Brasília.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Tool Handling Across Frameworks

| Aspect | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| Tool decorator | `@function_tool` | `@tool` (langchain_core) | `@tool` (crewai_tools) |
| Tool description | From docstring | From docstring | From docstring |
| How agent gets tools | `tools=[]` on Agent | `tools=[]` on create_react_agent | `tools=[]` on Agent |
| Tool call loop | Built into Runner | Built into create_react_agent | Built into Crew |
| Can you see tool calls? | `result.raw_responses` | Full message trace | `verbose=True` output |

**Key insight:** All three frameworks handle tool calling the same way under the hood:
1. Agent receives question
2. Agent decides which tool to use
3. Framework calls the tool function
4. Agent receives the tool result and formulates the final answer

The difference is in how much of this loop you can see and control.

## Key Takeaway

Tools are what turn a chatbot into an agent. The pattern is universal:
1. Define a function
2. Decorate it (framework-specific)
3. Pass it to the agent
4. The agent decides when to use it

LangGraph gives you the most visibility into the tool-calling loop (you can see
every intermediate message). OpenAI SDK abstracts it away. CrewAI handles it
internally with verbose logging.

---

